What is micrograd?
Micrograd is basically an autograd engine. It's a scalar-valued autograd engine.

Autograd is short for automatic gradient. And really what it does is it implements backpropagation.

Backpropagation is this algorithm that allows you to efficiently evaluate the gradient of some kind of a loss function with respect to the weights of neural network. And what that allows us to do then is we can iteratively tune the weights of that neural network to minimize the loss function and therefore improve the accuracy of the network.

So backpropagation would be at the mathematical core of and modern deep neural network library, like Pytorch or JAX.

Neuarl networks are just a mathematical expression they take the input data as an input and they take the weights of a neural network as an input and it's a mathematical expression and your output is predictions of your neural net or the loss function.

## micrograd

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

Let's define a function, a scalar-valid function.

It just takes a single scalar x and returns a single scalar y.

In [ ]:
def f(x):
    return 3*x**2 - 4*x + 5

In [ ]:
f(3.0)

: 

In [ ]:
xs = np.arange(-5, 5, 0.25)
xs
ys = f(xs)
ys
plt.plot(xs, ys)

So now I'd like to think through what is the derivative of this function at any single input point x?

What is the derivative at different points x of this function?

No one in neural networks actually writes out the expression for the neural network net. It would be a massive expression. And so we're not going to take this kind of symbolic approach.

Instead, What I'd like to look at the definition of derivative and just make sure that we really understand what derivative is measuring, what it's telling you about the function.

The definiton of derivative is the limit as 'h' goes to zero of 'f of x+h' minus 'f of x' over h.
So basically what it's saying is if you slightly bump up at some point x by a small number h, how does the function respond? With what sensitivity does it respond? What is the slope at that point? Does the function go up or go down? And by how much? And that's the slope of that function, the slope of that response at that point.

And so we basically evaluate the derivative here numerically by taking a very small h. And the definition would ask us to take h to zero.

In [ ]:
h = 0.001
x = 3.0
f(x + h)

Do you expect 'f of x+h' to be slightly greater 20? or do you expect to be slightly lower than 20. 
If we slightly go positively, the function will respond positively. So you'd expect this to be slightly greater than 20.

And by how much is telling you the sort of the strength of that slope. The size of the slope.

In [ ]:
h = 0.001
x = 3.0
(f(x + h) - f(x))/h

So this is just a numerical approximation of the slope, because we have to make age very small to converge to the exact amount.

In [ ]:
h = 0.001
x = -3.0
(f(x + h) - f(x))/h

In [ ]:
h = 0.001
x = 2/3
(f(x + h) - f(x))/h

Let's get more complex case.

In [ ]:
a = 2.0
b = -3.0
c = 10.0
d = a*b + c
print(d)

Now I'd like to look at the derivatives of d with respect to a, b, and c.

In [ ]:
h = 0.0001

# input
a = 2.0
b = -3.0
c = 10.0

: 

In [ ]:
d1 = a*b + c
a += h
d2 = a*b + c

print('d1', d1)
print('d2', d2)
print('slope', (d2 - d1)/h)

In [ ]:
d1 = a*b + c
b += h
d2 = a*b + c

print('d1', d1)
print('d2', d2)
print('slope', (d2 - d1)/h)

In [ ]:
d1 = a*b + c
c += h
d2 = a*b + c

print('d1', d1)
print('d2', d2)
print('slope', (d2 - d1)/h)

So, now we have some intuitive sense of what this derivative is telling you about the function.

And we'd like to move to neural networks.

Neural networks will be pretty massive expressions, mathematical expression.

So we need some data structures that maintain these expressions.

In [ ]:
class Value:

    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f'Value(data={self.data})'
    # what the repr is doing is it's providing us a way to point out like a nice looking expression in Python

    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), '+')
        return out
    
    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other), '*')
        return out
    



    

a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)
a + b
a*b
a*b + c
d = a*b + c
d
d._prev
d._op

but now python doesn't know how to add two value object, so we have to tell it.

In [ ]:
class Value:

    def __init__(self, data):
        self.data = data
    
    def __repr__(self):
        return f'Value(data={self.data})'

    def __add__(self, other):
        out = Value(self.data + other.data)
        return out

In [ ]:
a = Value(2.0)
b = Value(-3.0)
a + b
# a.__add__(b)

a.__add__(b) That's what will happen internally, so 'b' will be 'other' and 'self' will be 'a'.

So what we're going to return is a new value object, and it's just going to be wrapping the plus of their data, but remember now because data is the actual like python number so this operator here is just the typical floating point plus addition now, it's not an addition of value objects, and will return a new value.

Let's now implement multiply

In [ ]:
class Value:

    def __init__(self, data):
        self.data = data
    
    def __repr__(self):
        return f'Value(data={self.data})'

    def __add__(self, other):
        out = Value(self.data + other.data)
        return out
    
    def __mul__(self, other):
        out = Value(self.data * other.data)
        return out

In [ ]:
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)
a*b
# a.__mul__(b)
a*b + c
# (a.__mul__(b)).__add__(c)

So now we are missing is the connected tissue of this expression.

We want to keep these expression graphs. So we need to know and keep pointers about what values produce what other values.

So here we are going to introduce a new variable, which we'll call 'children', and by defult it will be an empty tuple.

And then we're going to keep a slightly different variable in the class, which we will call 'underscore prev', which will be the 'set of children'. 

In [ ]:
class Value:

    def __init__(self, data, _children=()):
        self.data = data
        self._prev = set(_children)
        
    def __repr__(self):
        return f'Value(data={self.data})'

    def __add__(self, other):
        out = Value(self.data + other.data, (self, other))
        return out
    
    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other))
        return out

so now when we creat a value like 'a = Value(2.0)' with a constructor, children will be empty and prep will be the empty set.

But when we are creating a value through addition or multiplication, we're going to feed it in the children of this value, which in this case is self and other.

In [ ]:
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)
d = a*b + c
d
d._prev

So we know now the children of every single value, but we don't know what operation created this value.

So we need one more element here, let's call it 'underscore op'.

In [ ]:
class Value:

    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self._prev = set(_children)
        self._op = _op
        
    def __repr__(self):
        return f'Value(data={self.data})'

    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), '+')
        return out
    
    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other), '*')
        return out

so now we not just have d._prev, we also have d._op

In [ ]:
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)
d = a*b + c
d
d._op